# Explanation

## Configuration

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.ConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

## Data

In [ ]:
DIR_DATA = '../.data/split'
CONDITIONS = ('neutral','stress')
DURATIONS = (0,)
CLASS_NAMES = ('understanding', 'confusion')

from trainer import DataProviderInterParticipant
data = DataProviderInterParticipant(DIR_DATA, CONDITIONS, DURATIONS, CLASS_NAMES)

In [ ]:
from utils.csv import csv_col_dict
ACTION_UNITS = csv_col_dict(
    conf['Features']['action-units'].strip().split('\n')
)
FEATRUE_LABELS = tuple(
    map(
        lambda x: f"{x[0]} - {x[1]}", 
        zip(ACTION_UNITS['name'], ACTION_UNITS['description'])
    )
)

## SHAP

Load data for training and evaluation (neutral, stress and full)

In [ ]:
import shap
import numpy as np
from trainer import DataProvider
class Explainer:
    def __init__(self, data:DataProvider, n_samples = 500):
        # restructure data
        x_train, y_train, x_eval, y_eval = dict(), dict(), dict(), dict()
        ## condistioned data
        for c in data.conditions:
            samples = data._get(c, 0)
            x_train[c], y_train[c], _ = samples['train']
            x_eval[c], y_eval[c], _ = samples['test']

        ## full data
        x_train['full'] = np.concatenate(tuple(x_train.values()))
        y_train['full'] = np.concatenate(tuple(y_train.values()))
        x_eval['full'] = np.concatenate(tuple(x_eval.values()))
        y_eval['full'] = np.concatenate(tuple(y_eval.values()))

        ## Remove second dimension as it is 1 (single frame data)
        for k in x_train.keys():
            x_train[k] = x_train[k][:,0]
            y_train[k] = y_train[k][:,0]
            x_eval[k] = x_eval[k][:,0]
            y_eval[k] = y_eval[k][:,0]

        ## SHAP pobing data
        probe_data, sampling_data = dict(), dict()
        for k in x_eval.keys():
            sampling_data[k] = shap.kmeans(x_eval[k], 100)
            probe_data[k] = shap.sample(x_eval[k], n_samples)
        
        # Assign to class attributes
        self.x_train = x_train
        self.y_train = y_train
        self.x_eval = x_eval
        self.y_eval = y_eval
        self.probe_data = probe_data
        self.sampling_data = sampling_data
    
    @property
    def conditions(self):
        return tuple(self.probe_data.keys())

    def explain(self, model, condition, fit=True):
        # Fit model to data
        if fit:
            model.fit(self.x_train[condition], self.y_train[condition])
        # Compute shap values
        explainer = shap.KernelExplainer(
            lambda x: model.predict(x),
            self.sampling_data[condition],
            feature_names=FEATRUE_LABELS
        )
        # return explanation
        return explainer(self.probe_data[condition])

explainer = Explainer(data)
explainer.conditions
        

In [ ]:
def visualize(shap_values):
    for i, c in enumerate(data.class_labels):
        print(f"Class {c}")
        shap.plots.violin(
            shap_values[...,i],
            sort=False
        )

### MLP

In [ ]:
N_LAYERS = int(conf['Models']['mlp-hidden-layers'])
N_NEURONS = int(conf['Models']['mlp-layer-neurons'])

from models import MultiLabelMLP
model = MultiLabelMLP((N_NEURONS,) * N_LAYERS)

In [ ]:
shap_mlp = {c:explainer.explain(model, c) for c in explainer.conditions}

In [ ]:
visualize(shap_mlp['neutral'])

In [ ]:
visualize(shap_mlp['stress'])

In [ ]:
visualize(shap_mlp['full'])

### XGBoost

In [ ]:
import xgboost as xgb
N_ESTIMATORS_XGB = int(conf['Models']['xgb-estimators'])
model = xgb.XGBClassifier(eval_metric='logloss', n_estimators=N_ESTIMATORS_XGB, device='cuda')
shap_xgb = {c:explainer.explain(model, c) for c in explainer.conditions}

In [ ]:
visualize(shap_xgb['neutral'])

In [ ]:
visualize(shap_xgb['stress'])

In [ ]:
visualize(shap_xgb['full'])

### Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
N_ESTIMATORS_RF = int(conf['Models']['rf-estimators'])
model = RandomForestClassifier(N_ESTIMATORS_RF, n_jobs=16)
shap_rf = {c:explainer.explain(model, c) for c in explainer.conditions}

In [ ]:
visualize(shap_rf['neutral'])

In [ ]:
visualize(shap_rf['stress'])

In [ ]:
visualize(shap_rf['full'])

In [ ]:
import pickle
for n, shap_valuse in (('mlp', shap_mlp), ('xgb', shap_xgb), ('rf', shap_rf)):
    for k in shap_valuse.keys():
        with open(f"../.data/shap/{n}_{k}.pkl", mode='w+b') as file:
            pickle.dump(shap_valuse[k], file)

In [ ]:
with open('../.data/shap/mlp_neutral.pkl', mode='rb') as file:
    shap_values = pickle.load(file)
visualize(shap_values)

In [ ]:
import pickle
shap_mlp = dict()
for s in ('neutral', 'stress', 'full'):
    with open(f'../.data/shap/mlp_{s}.pkl', mode='rb') as file:
        shap_mlp[s] = pickle.load(file)

In [ ]:
shap_rf = dict()
for s in ('neutral', 'stress', 'full'):
    with open(f'../.data/shap/rf_{s}.pkl', mode='rb') as file:
        shap_rf[s] = pickle.load(file)

In [ ]:
shap_xgb = dict()
for s in ('neutral', 'stress', 'full'):
    with open(f'../.data/shap/xgb_{s}.pkl', mode='rb') as file:
        shap_xgb[s] = pickle.load(file)

In [ ]:
import numpy as np
values = dict()
for s in ('neutral', 'stress', 'full'):
    values[s] = np.concatenate((
        shap_mlp[s].values,
        shap_rf[s].values,
        shap_xgb[s].values
    ))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
_, axs = plt.subplots(2,3, figsize=(15,10), sharex=True, sharey=True)
cols = ACTION_UNITS['name']
for s, ax in enumerate(axs):
    for a, c in zip(ax, ('neutral', 'stress', 'full')):
        corr = pd.DataFrame(values[c][...,s], columns=cols).corr()
        a.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
        a.set_xticks(ticks=range(len(cols)), labels=cols, rotation=90)
        a.set_yticks(ticks=range(len(cols)), labels=cols)
        if s == 0:
            a.set_title(c)
axs[0,0].set_ylabel('understanding')
axs[1,0].set_ylabel('confusion')
plt.suptitle('Feature importance correlation across MLP, XGBoost and Random Forest')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
def importance_comparison(domain):
    # define labels
    idx = ('mlp', 'rf', 'xgb')
    cols = ACTION_UNITS['name']

    # restructure data
    values = np.abs(np.array(tuple(
        s[domain].values.mean(axis=0)
        for s in (shap_mlp, shap_rf, shap_xgb)
    )))

    df_u = pd.DataFrame(values[...,0], index=idx, columns=cols).T
    df_c = pd.DataFrame(values[...,1], index=idx, columns=cols).T

    ax = (df_u).plot(kind='bar', ylim=(0,0.025), title=f"Feature importance for understanding")
    ax.hlines((0.5), -1, 18, color='red')
    ax = (df_c).plot(kind='bar', ylim=(0,0.025), title=f"Feature importance for confusion")
    ax.hlines((0.5), -1, 18, color='red')

importance_comparison('neutral')

In [ ]:
importance_comparison('stress')

In [ ]:
importance_comparison('full')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
def shap_statistics(shap_values, name):
    # define labels
    idx = tuple(shap_values.keys())
    cols = ACTION_UNITS['name']

    # restructure data
    values = np.array(tuple(
        s.values
        for s in shap_values.values()
    ))

    # means
    means = np.abs(values.mean(axis=1))
    df_u = pd.DataFrame(means[...,0], index=idx, columns=cols).T
    df_c = pd.DataFrame(means[...,1], index=idx, columns=cols).T
    df_u.plot(kind='bar', ylim=(0,0.025), title=f"{name} feature importance for understanding")
    df_c.plot(kind='bar', ylim=(0,0.025), title=f"{name} feature importance for confusion")

    #print(df_u)
    #corr = pd.DataFrame(values[0,:,:,0]).corr()
    #plt.matshow(df_u.T.corr(), cmap='coolwarm', vmin=-1, vmax=1)
    #plt.xticks(ticks=range(len(cols)), labels=cols, rotation=90)
    #plt.yticks(ticks=range(len(cols)), labels=cols)

shap_statistics(shap_mlp, 'MLP')

In [ ]:
shap_statistics(shap_mlp, 'MLP')

In [ ]:
shap_statistics(shap_xgb, 'XGBoost')

In [ ]:
shap_statistics(shap_rf, 'Random Forests')